In [1]:
import os
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score
)

from CellClassificationEvaluator import CellClassificationEvaluator

/home/a2236013/.local/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/a2236013/.local/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (
/home/a2236013/anaconda3/lib/python3.9/site-packages/torch/cuda/__init__.py:51: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
if __name__ == "__main__":
    # all data are stored in this relative path
    data_dir = "../example_data/for_cell_annotation/"
    
    # Target dataset to evaluate
    tissue_name = "Prostate"
    method_name = "Example_of_Method"  # Replace with your actual method name
    device_num = "cuda:0"          # Modify according to your GPU setup, or use "cpu"

    print(f"Starting evaluation on the {tissue_name} dataset...")

    # 1. Initialize the evaluator (both paths point to the same directory)
    evaluator = CellClassificationEvaluator(
        base_data_path=data_dir,   # Directory containing embedding files
        label_data_path=data_dir,  # Directory containing label files
        device=device_num,
        seed=42,
        num_epochs=50,            # For quick testing, you can reduce this (e.g., 10)
        lr=1e-4,
        patience=5
    )

    # 2. Run five-fold cross-validation
    metrics_res = evaluator.run_5_fold(tissue_name=tissue_name, method_name=method_name)

    # 3. Print and save results
    df_results = pd.DataFrame(metrics_res)
    print("\n✅ Five-fold cross-validation results:")
    print(df_results)
    
    # Compute mean and standard deviation for each metric (optional, for overall performance summary)
    summary = df_results[["Macro-F1", "Macro-AUROC", "Macro-AUPRC"]].agg(['mean', 'std'])
    print("\n📊 Overall performance statistics:")
    print(summary)


Starting evaluation on the Prostate dataset...



✅ Five-fold cross-validation results:
   Macro-F1  Macro-AUROC  Macro-AUPRC    Tissue  Fold        Method_name
0  0.880095     0.997039     0.932453  Prostate     1  Example_of_Method
1  0.865395     0.994861     0.930443  Prostate     2  Example_of_Method
2  0.896229     0.998366     0.951255  Prostate     3  Example_of_Method
3  0.884557     0.996025     0.926561  Prostate     4  Example_of_Method
4  0.890739     0.997260     0.931183  Prostate     5  Example_of_Method

📊 Overall performance statistics:
      Macro-F1  Macro-AUROC  Macro-AUPRC
mean  0.883403     0.996710     0.934379
std   0.011778     0.001327     0.009687
